# Train CRNN captcha — Colab / Kaggle

Train 14 model CRNN+CTC (một model cho mỗi nguồn captcha trong `pass/`).

**Ba điều quan trọng nhất:**

1. **Zip dataset tại nguồn.** Copy 26k file nhỏ qua Drive mount chậm kinh khủng; một
   `pass.zip` ~200 MB mất ~30 s. Upload private Kaggle dataset rồi `unzip` ra disk
   instance. **Không bao giờ train trực tiếp trên Drive mount.**
2. **Loop 14 nhóm độc lập, copy `runs/` lên Drive sau mỗi nhóm** — disconnect không
   mất quá 1 nhóm. Loop tự resume vì nhóm nào đã có `best.pt` thì bị skip.
3. **`inspect_data` là cổng bắt buộc** trước mọi lần train. Nó verify `T >= 2L+1`
   cho từng nhóm; nếu bỏ qua và một nhóm vi phạm, CTCLoss trả `inf` bị
   `zero_infinity` che mất và model âm thầm không học gì.


## 1. GPU + version


In [ ]:
!nvidia-smi -L || echo "NO GPU -- training will be very slow"
import sys, torch, platform
print("python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"


## 2. Lấy project code

Chọn **một** trong hai cách, rồi set `PROJECT`:

- **A.** `git clone` repo của bạn (nếu đã push).
- **B.** Upload một zip **chỉ chứa code** (`captcha/`, `app.py`, `configs/`,
  `requirements*.txt`, `tests/`, `colab_train.ipynb`) — **đừng** kèm `pass/`.


In [ ]:
import os, zipfile
PROJECT = "/content/scan-captcha-project"     # <-- sửa nếu cần

# --- A. git clone ---------------------------------------------------------
# !git clone https://github.com/<user>/<repo>.git {PROJECT}

# --- B. upload zip code ---------------------------------------------------
# from google.colab import files
# up = files.upload()                        # chọn code.zip
# os.makedirs(PROJECT, exist_ok=True)
# with zipfile.ZipFile(next(iter(up)), "r") as z: z.extractall(PROJECT)

print(sorted(os.listdir(PROJECT))[:20])
assert os.path.isdir(f"{PROJECT}/captcha"), f"chưa có code ở {PROJECT}"


## 3. Cài dependencies


In [ ]:
%cd {PROJECT}
!pip install -q -r requirements.txt -r requirements-optional.txt
!python -c "import torch, cv2, yaml, onnxruntime, flask; print('deps OK')"


## 4. Lấy dataset (zip tại nguồn, unzip ra disk instance)

Tạo `pass.zip` **một lần ở máy local**:

```powershell
Compress-Archive -Path pass\* -DestinationPath pass.zip
```

Rồi upload làm **private Kaggle Dataset** (khuyến nghị — nhanh và ổn định), hoặc để
trên Drive.

> Trên Kaggle Notebooks: attach dataset luôn thì **zero copy**, tốt nhất —
> `DATA = "/kaggle/input/<dataset>/pass"`.


In [ ]:
DATA = "/content/pass"          # thư mục chứa 25,976 file .png

# --- Kaggle dataset (khuyến nghị) -----------------------------------------
# !pip install -q kaggle
# from google.colab import files; files.upload()      # kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <user>/captcha-pass -p /content --unzip
# DATA = "/content/pass"

# --- Drive -----------------------------------------------------------------
# from google.colab import drive; drive.mount("/content/drive")
# !cp /content/drive/MyDrive/pass.zip /content/ && cd /content && unzip -q pass.zip -d pass

import os
n = len([f for f in os.listdir(DATA) if f.lower().endswith(".png")]) if os.path.isdir(DATA) else 0
print(f"{DATA}: {n} PNG")
assert n == 25976, f"expected 25976 images, found {n}"


## 5. Cổng sanity — BẮT BUỘC

Assert đúng 14 nhóm, tổng 25,976, `T >= 2L+1` mọi nhóm, và polarity sau normalize
đúng. Nếu cell này fail thì **đừng train** — sửa nguyên nhân trước.


In [ ]:
!python -m captcha.inspect_data --data-dir {DATA} \
    --write-config configs/groups.yaml --json-out out/report.json


Cell trên in bảng 14 nhóm. Đọc kỹ:

- `chk` phải là `OK` mọi dòng.
- `rev = REVIEW` ở `g_600x120` / `g_250x40` / `g_150x30`: drop > 10%, đuôi nhiều khả
  năng là charset thật. Đổi `charset_mode: core` → `full` trong `configs/groups.yaml`
  rồi train lại để so sánh, hoặc xem ảnh bằng `--dump-tail-samples`.


In [ ]:
!python -m captcha.inspect_data --data-dir {DATA} \
    --dump-tail-samples out/tail --per-group 12
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("out/tail/g_80x30__*.png"))[:6]:
    display(Image(p, width=340))
print("Nếu ký tự đuôi trông giống hệt một ký tự lõi -> đó là nhãn sai, không phải charset mới.")


## 6. Train


In [ ]:
import subprocess, sys, shutil, time, yaml
from pathlib import Path

EPOCHS   = 60
BATCH    = 128
WORKERS  = 2
DRIVE    = Path("/content/drive/MyDrive/captcha_runs")   # đổi nếu muốn persist

cfg = yaml.safe_load(Path("configs/groups.yaml").read_text(encoding="utf-8"))
groups = [g["group_id"] for g in cfg["groups"]]
groups.sort(key=lambda g: -next(int(s[0]) for s in
            next(x["sizes"] for x in cfg["groups"] if x["group_id"] == g)))
print(f"{len(groups)} groups:", groups)


In [ ]:
runs = Path("runs")
for i, gid in enumerate(groups, 1):
    if (runs / gid / "best.pt").exists():
        print(f"[{i}/{len(groups)}] {gid}: already trained, skip")
        continue
    t0 = time.time()
    print(f"[{i}/{len(groups)}] training {gid} ...", flush=True)
    r = subprocess.run(
        [sys.executable, "-m", "captcha.train", "--group", gid,
         "--data-dir", DATA, "--epochs", str(EPOCHS),
         "--batch-size", str(BATCH), "--num-workers", str(WORKERS)],
        capture_output=True, text=True)
    tail = (r.stdout or "")[-2500:]
    print(tail)
    if r.returncode != 0:
        print("FAILED:", gid, "\n", (r.stderr or "")[-2500:])
    if DRIVE.parent.exists():
        shutil.copytree(runs / gid, DRIVE / gid, dirs_exist_ok=True)
    print(f"  {gid} done in {(time.time() - t0) / 60:.1f} min\n", flush=True)


## 7. Đánh giá


In [ ]:
import csv, json, subprocess, sys
from pathlib import Path

rows = []
for gid in groups:
    if not (Path("runs") / gid / "best.pt").exists():
        continue
    j = Path("out") / f"eval_{gid}.json"
    j.parent.mkdir(exist_ok=True)
    r = subprocess.run([sys.executable, "-m", "captcha.evaluate", "--group", gid,
                        "--data-dir", DATA, "--split", "test",
                        "--json-out", str(j)], capture_output=True, text=True)
    if r.returncode != 0:
        print("eval failed:", gid, r.stderr[-800:])
        continue
    rep = json.loads(j.read_text())
    m = rep["metrics"]
    n = next(x["n"] for x in json.loads(
        (Path("out") / "report.json").read_text()) if x["group_id"] == gid)
    rows.append((gid, n, m["exact_match"], m["char_acc"], m["cer"]))
    print(f"{gid:12s} EM {m['exact_match']:6.2%}  char {m['char_acc']:6.2%}  "
          f"CER {m['cer']:.4f}")

total = sum(r[1] for r in rows)
wem = sum(r[1] * r[2] for r in rows) / total if total else 0
wch = sum(r[1] * r[3] for r in rows) / total if total else 0
print(f"\nWEIGHTED over {total} images: exact-match {wem:.2%}, per-char {wch:.2%}")
with open("results.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["group", "n", "exact_match", "char_acc", "cer"])
    w.writerows(rows)


**Đọc bảng confusion trước khi kết luận model dở.** Nếu top toàn `c↔C`, `x↔X`,
`0↔o` thì đó là **nhiễu nhãn** (nhãn do người gán), không phải lỗi model — và là
trần thật của exact-match.


In [ ]:
import json, subprocess, sys
gid = "g_600x120"          # nhóm drop cao nhất
print(subprocess.run([sys.executable, "-m", "captcha.evaluate", "--group", gid,
                      "--data-dir", DATA, "--split", "test"],
                     capture_output=True, text=True).stdout)


## 8. Export ONNX + kiểm tra parity


In [ ]:
!python -m captcha.export_onnx --all
# exporter tự chạy cùng batch qua torch và onnxruntime, fail nếu lệch > 1e-4.
!ls -la runs/*/model.onnx | head -20


In [ ]:
# ONNX phải cho cùng chuỗi như torch backend
import glob, subprocess, sys
sample = sorted(glob.glob(f"{DATA}/*.png"))[0]
for backend in ("torch", "onnx"):
    out = subprocess.run([sys.executable, "-m", "captcha.predict", "--backend", backend,
                          "--data-dir", DATA, "--image", sample],
                         capture_output=True, text=True).stdout.strip()
    print(f"{backend:6s} {out}")


## 9. Biểu đồ loss / accuracy


In [ ]:
from pathlib import Path
from captcha.plotting import plot_history
from IPython.display import Image, display

for gid in groups:
    h = Path("runs") / gid / "history.csv"
    if not h.exists():
        continue
    for p in plot_history(h, Path("runs") / gid / "plots", gid, save=True):
        display(Image(str(p), width=760))


## 10. Đóng gói kết quả để tải về


In [ ]:
import shutil, os
shutil.make_archive("/content/captcha_models", "zip", "runs")
shutil.copy("configs/groups.yaml", "/content/groups.yaml")
shutil.copy("results.csv", "/content/results.csv")
print("ZIP:", os.path.getsize("/content/captcha_models.zip") / 1e6, "MB")
try:
    from google.colab import files
    files.download("/content/captcha_models.zip")
except Exception as e:
    print("tải thủ công:", e)


## Bước tiếp theo

1. Giải nén `captcha_models.zip` vào `runs/` ở máy local (giữ nguyên tên thư mục nhóm).
2. `python app.py --backend onnx` → mở web demo.
3. `python -m captcha.predict --image <ảnh mới>` → CLI.

Nhóm nào accuracy thấp: xem bảng confusion trước. Nếu là nhiễu nhãn thì không sửa
được bằng model; nếu là style trộn (như `g_250x40`) thì tách nhóm theo style rồi
train lại.
